# 01. Exploratory Data Analysis (EDA) — Vehicle Casting Defect Detection

**Project Goal:** Automated visual inspection of cast metal components (submersible pump impellers) to detect manufacturing defects.

**Dataset:** Kaggle "Casting product image data for quality inspection" (`ravirajsinh45/real-life-industrial-dataset-of-casting-product`).
- Images: ~300x300 grayscale JPEG captures.
- Classes: `def_front` (Defective casting) vs. `ok_front` (Flawless casting).

## 0. Google Colab Environment Setup
Run this section if running in **Google Colab**. It clones/pulls the repository, installs dependencies, downloads the dataset from Kaggle, and verifies directory layout with `src/utils.py`.

In [ ]:
# Step 0a: Clone or pull the GitHub repository
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_dir = 'vehicle-defect-detection'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/annugupta25/vehicle-defect-detection.git
        %cd vehicle-defect-detection
    else:
        %cd vehicle-defect-detection
        !git pull origin main
    
    # Ensure project root is on sys.path
    if '.' not in sys.path:
        sys.path.insert(0, '.')
    print("Current working directory:", os.getcwd())
else:
    print("Running locally. Project root confirmed.")

In [ ]:
# Step 0b: Install dependencies pinned in requirements.txt
if IN_COLAB:
    !pip install -q -r requirements.txt
    print("Dependencies installed successfully!")
else:
    print("Ensure your local virtual environment has dependencies installed via: pip install -r requirements.txt")

In [ ]:
# Step 0c: Kaggle API authentication & dataset download
dataset_target = os.path.join("data", "casting_data", "casting_data")

if not os.path.exists(dataset_target):
    print("Dataset not found. Initiating download...")
    if IN_COLAB:
        from google.colab import files
        if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
            print("Please upload your 'kaggle.json' API token file:")
            uploaded = files.upload()
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        
        !kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product
        !mkdir -p data
        !unzip -q -o real-life-industrial-dataset-of-casting-product.zip -d data/
        print("Dataset extracted to data/")
    else:
        print("Please place the unzipped dataset in 'data/casting_data/casting_data/' locally.")
else:
    print(f"Dataset already present at: {dataset_target}")

In [ ]:
# Step 0d: Verify folder layout with src/utils.py
from src.utils import check_dataset_paths

status = check_dataset_paths("data/casting_data/casting_data")
print("Dataset path check results:")
for k, v in status.items():
    print(f"  - {k}: {v}")

if status["train_exists"] and status["test_exists"]:
    print("\n Dataset directory structure verified successfully! Ready for EDA.")
else:
    print("\n⚠️ Dataset directories missing. Please check the paths above before proceeding.")

## 1. Imports & Visualization Setup

In [ ]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Visual configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

DATA_DIR = Path("data/casting_data/casting_data")
FIGURES_DIR = Path("reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset root:   {DATA_DIR}")
print(f"Figures export: {FIGURES_DIR}")

## 2. Class Balance Analysis

Inspect class distribution in both training and test directories to assess class balance.

In [ ]:
def get_class_counts(data_root):
    records = []
    for split in ["train", "test"]:
        for label in ["def_front", "ok_front"]:
            class_name = "Defective" if "def" in label else "OK"
            folder = data_root / split / label
            if folder.exists():
                count = len(list(folder.glob("*.jpeg")) + list(folder.glob("*.jpg")) + list(folder.glob("*.png")))
            else:
                count = 0
            records.append({"Split": split.capitalize(), "Class": class_name, "Folder": label, "Count": count})
    return pd.DataFrame(records)

df_counts = get_class_counts(DATA_DIR)
display(df_counts)

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Train split
train_subset = df_counts[df_counts["Split"] == "Train"]
if train_subset["Count"].sum() > 0:
    sns.barplot(data=train_subset, x="Class", y="Count", palette=["#e74c3c", "#2ecc71"], ax=axes[0])
    axes[0].set_title("Training Set: Class Distribution", fontweight="bold", fontsize=13)
    axes[0].set_ylabel("Number of Images")
    for p in axes[0].patches:
        height = int(p.get_height())
        total = train_subset['Count'].sum()
        pct = (height / total) if total > 0 else 0
        axes[0].annotate(f"{height}\n({pct:.1%})", 
                        (p.get_x() + p.get_width() / 2., max(height / 2, 5)),
                        ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# Test split
test_subset = df_counts[df_counts["Split"] == "Test"]
if test_subset["Count"].sum() > 0:
    sns.barplot(data=test_subset, x="Class", y="Count", palette=["#e74c3c", "#2ecc71"], ax=axes[1])
    axes[1].set_title("Test Set: Class Distribution", fontweight="bold", fontsize=13)
    axes[1].set_ylabel("Number of Images")
    for p in axes[1].patches:
        height = int(p.get_height())
        total = test_subset['Count'].sum()
        pct = (height / total) if total > 0 else 0
        axes[1].annotate(f"{height}\n({pct:.1%})", 
                        (p.get_x() + p.get_width() / 2., max(height / 2, 5)),
                        ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.suptitle("Casting Product Dataset — Class Balance Overview", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
fig_path = FIGURES_DIR / "class_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Figure saved to: {fig_path}")
plt.show()

## 3. Sample Images Inspection (Defective vs OK)

Visual inspection of sample parts to observe cracks, pinholes, and voids.

In [ ]:
def get_sample_images(data_root, n_samples=4):
    samples = {"Defective": [], "OK": []}
    def_files = (list((data_root / "train" / "def_front").glob("*.jpeg")) + 
                 list((data_root / "train" / "def_front").glob("*.jpg")))[:n_samples]
    ok_files = (list((data_root / "train" / "ok_front").glob("*.jpeg")) + 
                list((data_root / "train" / "ok_front").glob("*.jpg")))[:n_samples]
    
    for f in def_files:
        samples["Defective"].append(Image.open(f).convert("RGB"))
    for f in ok_files:
        samples["OK"].append(Image.open(f).convert("RGB"))
    return samples

sample_dict = get_sample_images(DATA_DIR, n_samples=4)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, img in enumerate(sample_dict["Defective"]):
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Defective #{i+1}", color="#c0392b", fontweight="bold")
    axes[0, i].axis("off")

for i, img in enumerate(sample_dict["OK"]):
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"OK #{i+1}", color="#27ae60", fontweight="bold")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("DEFECTIVE", fontsize=13, fontweight="bold", color="#c0392b")
axes[1, 0].set_ylabel("OK / NORMAL", fontsize=13, fontweight="bold", color="#27ae60")
plt.suptitle("Sample Casting Inspection: Defective (Top) vs. OK (Bottom)", fontsize=15, fontweight="bold", y=0.98)
plt.tight_layout()
fig_path = FIGURES_DIR / "sample_images.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Figure saved to: {fig_path}")
plt.show()

## 4. Image Dimensions & Aspect Ratio Analysis

In [ ]:
image_metadata = []
for split in ["train", "test"]:
    for label in ["def_front", "ok_front"]:
        folder = DATA_DIR / split / label
        files = (list(folder.glob("*.jpeg")) + list(folder.glob("*.jpg")))[:75]
        for f in files:
            with Image.open(f) as img:
                w, h = img.size
                image_metadata.append({
                    "Split": split,
                    "Class": "Defective" if "def" in label else "OK",
                    "Width": w,
                    "Height": h,
                    "Aspect_Ratio": round(w / h, 3),
                })

df_meta = pd.DataFrame(image_metadata)
if len(df_meta) > 0:
    display(df_meta[["Width", "Height", "Aspect_Ratio"]].describe())
    
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(data=df_meta, x="Width", y="Height", hue="Class", palette=["#e74c3c", "#2ecc71"], alpha=0.7, ax=ax)
    ax.set_title("Image Dimensions Distribution (Width vs Height)", fontweight="bold", fontsize=12)
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Height (px)")
    plt.tight_layout()
    fig_path = FIGURES_DIR / "image_size_distribution.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {fig_path}")
    plt.show()

## 5. Pixel-Intensity Distribution Analysis

In [ ]:
def collect_pixel_intensities(data_root, n_images=50):
    def_pixels = []
    ok_pixels = []
    
    def_files = (list((data_root / "train" / "def_front").glob("*.jpeg")) + 
                 list((data_root / "train" / "def_front").glob("*.jpg")))[:n_images]
    ok_files = (list((data_root / "train" / "ok_front").glob("*.jpeg")) + 
                list((data_root / "train" / "ok_front").glob("*.jpg")))[:n_images]
    
    for f in def_files:
        arr = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if arr is not None:
            def_pixels.extend(arr.flatten()[::20])
    for f in ok_files:
        arr = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if arr is not None:
            ok_pixels.extend(arr.flatten()[::20])
            
    return np.array(def_pixels), np.array(ok_pixels)

def_pixels, ok_pixels = collect_pixel_intensities(DATA_DIR, n_images=50)

if len(def_pixels) > 0 and len(ok_pixels) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.kdeplot(def_pixels, label="Defective (def_front)", color="#e74c3c", fill=True, alpha=0.3, lw=2, ax=ax)
    sns.kdeplot(ok_pixels, label="OK (ok_front)", color="#2ecc71", fill=True, alpha=0.3, lw=2, ax=ax)
    
    ax.set_title("Pixel Intensity Distribution (KDE): Defective vs. OK Parts", fontweight="bold", fontsize=13)
    ax.set_xlabel("Pixel Grayscale Intensity (0 = Black, 255 = White)", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_xlim(0, 255)
    ax.legend(fontsize=11)
    
    plt.tight_layout()
    fig_path = FIGURES_DIR / "pixel_intensity_distribution.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {fig_path}")
    plt.show()